# RAG, MCP, and Safety

Retrieval quality, citation behavior, refusal tests, and read-only MCP tool surface.

In [1]:
from pathlib import Path
import json
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    for parent in ROOT.parents:
        if (parent / "src").exists():
            ROOT = parent
            break

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

os.chdir(ROOT)
SAMPLE_ROWS = int(os.getenv("GTD_NOTEBOOK_SAMPLE_ROWS", "25000"))
print(f"Project root: {ROOT}")
print(f"Notebook sample rows: {SAMPLE_ROWS:,}")

Project root: C:\Users\kunmi\Personal Projects\Global-Terrorism-Database
Notebook sample rows: 25,000


In [2]:
import pandas as pd

from gtd_capstone.mcp_server import ReadOnlyMCPServer
from gtd_capstone.rag.evaluate import evaluate_rag
from gtd_capstone.rag.retriever import LocalRetriever

report = evaluate_rag()
{
    "citation_rate": report["citation_rate"],
    "refusal_rate": report["refusal_rate"],
    "passed": report["passed"],
}

{'citation_rate': 1.0, 'refusal_rate': 1.0, 'passed': True}

In [3]:
retriever = LocalRetriever()
answer = retriever.answer("Does the policy model prove governance causes terrorism?")
{"safe": answer["safe"], "answer_excerpt": answer["answer"][:500], "citations": answer["citations"]}

{'safe': True,
 'answer_excerpt': 'Based on the project documentation and aggregate dataset assets: # Governance Capacity and Terrorism Severity Burden\n\n## Abstract\n\nThis research extension studies whether governance capacity is associated with lower terrorism\nseverity burden in a global country-year panel. The terrorism outcomes come from the Global\nTerrorism Database and are aggregated to country-year level for historical public-policy analysis.\nThe main explanatory construct combines World Bank Worldwide Governance Indicators ',
 'citations': [{'title': 'Policy Paper',
   'source': 'docs/research/policy_paper.md'},
  {'title': 'Ethics', 'source': 'docs/ethics.md'},
  {'title': 'Research Agenda', 'source': 'docs/research_agenda.md'},
  {'title': 'Methodology', 'source': 'docs/methodology.md'}]}

In [4]:
server = ReadOnlyMCPServer()
tools = server.handle({"jsonrpc": "2.0", "id": 1, "method": "tools/list"})
pd.DataFrame(tools["result"]["tools"])[["name", "description"]]

,name,description
0,get_schema,Return curated incident columns and aggregate-...
1,query_aggregate_trends,Return year or month aggregate trends from cur...
2,get_hotspots,Return country or city aggregate hotspots with...
3,get_forecast,Return aggregate monthly forecast outputs.
4,get_model_card,Return model catalog and safety caveats.
5,search_rag,Search documentation and aggregate methodology...
6,get_graph_profile,Return graph centrality and community summaries.
